# AIクラウド: Colab + Ollama + Gemma4

このノートブックは、Google Colab上でOllamaを起動し、`gemma4:26b` を呼び出すAPIを作ります。最後にCloudflare Quick Tunnelの `trycloudflare.com` URLを表示するので、AIクラウドの画面上部に貼ってください。

GitHub Pagesから使う場合は、AIクラウド側の形式を `Ollama`、モデルを `gemma4:26b` にします。Colab無料枠はリソース保証がなく、Webサービス用途には向かないため、公開デモや長時間運用ではなく、ノートブックを開いた状態でのテスト用として使ってください。

In [ ]:
# 設定
MODEL = "gemma4:26b"
PORT = 8000

# 本番寄りにするなら "https://あなたのユーザー名.github.io" に変えてください。
# まず動作確認だけなら "*" のままでOKです。
ALLOWED_ORIGINS = "*"

OLLAMA_BASE = "http://127.0.0.1:11434"
RUN_PUBLIC_CHAT_TEST = True

In [ ]:
# GPUとColab環境の確認
!nvidia-smi || true
!python --version
!df -h /content

In [ ]:
# Ollama / APIサーバー / cloudflared を準備
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install fastapi "uvicorn[standard]" httpx requests
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!ollama --version
!cloudflared --version

In [ ]:
# Ollamaをバックグラウンド起動し、gemma4:26bを取得
import os
import pathlib
import subprocess
import time
import requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_ORIGINS"] = ALLOWED_ORIGINS

old_ollama_proc = globals().get("ollama_proc")
if old_ollama_proc is not None and old_ollama_proc.poll() is None:
    old_ollama_proc.terminate()
    time.sleep(1)

ollama_log_path = pathlib.Path("/content/ollama.log")
ollama_log = ollama_log_path.open("w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

def wait_for_ollama(timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(f"{OLLAMA_BASE}/api/version", timeout=2)
            if response.ok:
                return response.json()
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"Ollamaが起動しませんでした。ログ: {ollama_log_path}")

print("Ollama:", wait_for_ollama())

pull_proc = subprocess.Popen(
    ["ollama", "pull", MODEL],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)
for line in pull_proc.stdout:
    print(line.rstrip())
if pull_proc.wait() != 0:
    raise RuntimeError(f"{MODEL} の取得に失敗しました。GPU/RAM/ディスク容量を確認してください。")

print("Model ready:", MODEL)

print("Warm-up: loading model into Ollama...")
warmup = requests.post(
    f"{OLLAMA_BASE}/api/chat",
    json={"model": MODEL, "messages": [{"role": "user", "content": "OKだけ返して"}], "stream": False, "keep_alive": "30m", "options": {"num_predict": 16}},
    timeout=600,
)
if warmup.status_code >= 400:
    raise RuntimeError(f"ウォームアップに失敗しました: {warmup.status_code} {warmup.text[:1000]}")
print("Warm-up response:", warmup.json().get("message", {}).get("content", "")[:120])

In [ ]:
%%writefile gemma4_bridge.py
import os
import re
import time
from typing import Any

import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

MODEL = os.environ.get("OLLAMA_MODEL", "gemma4:26b")
OLLAMA_BASE = os.environ.get("OLLAMA_BASE", "http://127.0.0.1:11434").rstrip("/")
origin_setting = os.environ.get("ALLOWED_ORIGINS", "*")
ALLOWED_ORIGINS = [origin.strip() for origin in origin_setting.split(",") if origin.strip()] or ["*"]

app = FastAPI(title="AI Cloud Gemma4 Colab Bridge")
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)

def text_from(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                parts.append(text_from(item.get("text") or item.get("content") or item.get("value") or ""))
            else:
                parts.append(text_from(item))
        return "".join(parts)
    if isinstance(value, dict):
        return text_from(value.get("text") or value.get("content") or value.get("value"))
    return str(value)

def clean_messages(messages: Any) -> list[dict[str, str]]:
    cleaned = []
    if not isinstance(messages, list):
        return cleaned
    for item in messages:
        if not isinstance(item, dict):
            continue
        role = item.get("role", "user")
        if role not in {"system", "user", "assistant", "tool"}:
            role = "user"
        content = text_from(item.get("content")).strip()
        if content:
            cleaned.append({"role": role, "content": content})
    return cleaned

def messages_from_custom(body: dict[str, Any]) -> list[dict[str, str]]:
    messages = clean_messages(body.get("history"))
    message = text_from(body.get("message") or body.get("prompt")).strip()
    if message:
        messages.append({"role": "user", "content": message})
    return messages

async def read_json(request: Request) -> dict[str, Any]:
    try:
        body = await request.json()
    except Exception as exc:
        raise HTTPException(status_code=400, detail="JSON body is required") from exc
    if not isinstance(body, dict):
        raise HTTPException(status_code=400, detail="JSON object is required")
    return body

async def ollama_chat(payload: dict[str, Any]) -> dict[str, Any]:
    timeout = httpx.Timeout(300.0, connect=30.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.post(f"{OLLAMA_BASE}/api/chat", json=payload)
    if response.status_code >= 400:
        raise HTTPException(status_code=response.status_code, detail=response.text[:4000])
    return response.json()

def clean_reply_text(text: str) -> str:
    text = re.sub(r"<\|channel\>thought[\s\S]*?<channel\|>", "", text)
    return text.replace("<|channel>final", "").strip()

def reply_text(data: dict[str, Any]) -> str:
    message = data.get("message")
    message_content = message.get("content") if isinstance(message, dict) else message
    return clean_reply_text(text_from(
        message_content
        or data.get("reply")
        or data.get("response")
        or data.get("text")
    ))

def make_payload(body: dict[str, Any], messages: list[dict[str, str]]) -> dict[str, Any]:
    if not messages:
        raise HTTPException(status_code=400, detail="message or messages is required")
    payload = {
        "model": body.get("model") or MODEL,
        "messages": messages,
        "stream": False,
        "keep_alive": body.get("keep_alive", "30m"),
    }
    for key in ("options", "format", "think", "tools"):
        if key in body:
            payload[key] = body[key]
    return payload

@app.get("/")
async def root():
    return {"ok": True, "service": "AI Cloud Gemma4 Colab Bridge", "model": MODEL}

@app.get("/health")
@app.get("/api/health")
async def health():
    async with httpx.AsyncClient(timeout=10) as client:
        version = (await client.get(f"{OLLAMA_BASE}/api/version")).json()
    return {"ok": True, "model": MODEL, "ollama": version}

@app.post("/api/chat")
async def api_chat(request: Request):
    body = await read_json(request)
    messages = clean_messages(body.get("messages")) or messages_from_custom(body)
    data = await ollama_chat(make_payload(body, messages))
    return JSONResponse(data)

@app.post("/chat")
@app.post("/api/gemma")
async def custom_chat(request: Request):
    body = await read_json(request)
    messages = clean_messages(body.get("messages")) or messages_from_custom(body)
    data = await ollama_chat(make_payload(body, messages))
    content = reply_text(data)
    return {"ok": True, "model": data.get("model", MODEL), "reply": content, "response": content, "message": {"role": "assistant", "content": content}, "raw": data}

@app.post("/v1/chat/completions")
async def openai_chat_completions(request: Request):
    body = await read_json(request)
    messages = clean_messages(body.get("messages"))
    data = await ollama_chat(make_payload(body, messages))
    content = reply_text(data)
    return {
        "id": f"chatcmpl-colab-{int(time.time() * 1000)}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": data.get("model", body.get("model") or MODEL),
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": data.get("done_reason", "stop")}],
        "usage": {},
    }


In [ ]:
# APIサーバーとCloudflare Quick Tunnelを起動
import os
import pathlib
import queue
import re
import subprocess
import threading
import time
import requests

os.environ["OLLAMA_MODEL"] = MODEL
os.environ["OLLAMA_BASE"] = OLLAMA_BASE
os.environ["ALLOWED_ORIGINS"] = ALLOWED_ORIGINS

for proc_name in ("api_proc", "tunnel_proc"):
    old_proc = globals().get(proc_name)
    if old_proc is not None and old_proc.poll() is None:
        old_proc.terminate()
        time.sleep(1)

api_log_path = pathlib.Path("/content/gemma4_bridge.log")
api_log = api_log_path.open("w")
api_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "gemma4_bridge:app", "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=api_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

def wait_for_api(timeout=60):
    deadline = time.time() + timeout
    url = f"http://127.0.0.1:{PORT}/health"
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=2)
            if response.ok:
                return response.json()
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"APIサーバーが起動しませんでした。ログ: {api_log_path}")

print("API:", wait_for_api())

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

lines = queue.Queue()
def read_tunnel_output():
    for line in tunnel_proc.stdout:
        lines.put(line)

threading.Thread(target=read_tunnel_output, daemon=True).start()

public_url = None
deadline = time.time() + 90
pattern = re.compile(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com")
while time.time() < deadline and public_url is None:
    try:
        line = lines.get(timeout=1)
    except queue.Empty:
        continue
    print(line.rstrip())
    match = pattern.search(line)
    if match:
        public_url = match.group(0)

if not public_url:
    raise RuntimeError("trycloudflare URLを取得できませんでした。cloudflaredの出力を確認してください。")

PUBLIC_URL = public_url

def wait_for_public_health(url, timeout=90):
    deadline = time.time() + timeout
    health_url = f"{url}/health"
    while time.time() < deadline:
        try:
            response = requests.get(health_url, timeout=10)
            if response.ok:
                return response.json()
            print("Public health pending:", response.status_code, response.text[:200])
        except Exception as exc:
            print("Public health pending:", exc)
        time.sleep(2)
    raise RuntimeError(f"Public tunnel health check failed: {health_url}")

public_health = wait_for_public_health(PUBLIC_URL)
cors_check = requests.options(
    f"{PUBLIC_URL}/api/chat",
    headers={"Origin": "https://example.github.io", "Access-Control-Request-Method": "POST", "Access-Control-Request-Headers": "Content-Type"},
    timeout=20,
)
print("Public health:", public_health)
print("CORS preflight:", cors_check.status_code, cors_check.headers.get("Access-Control-Allow-Origin"), cors_check.headers.get("Access-Control-Allow-Methods"))

if RUN_PUBLIC_CHAT_TEST:
    public_chat = requests.post(
        f"{PUBLIC_URL}/api/chat",
        json={"model": MODEL, "messages": [{"role": "user", "content": "日本語で短くOKと返して"}], "stream": False, "keep_alive": "30m", "options": {"num_predict": 16}},
        timeout=300,
    )
    if public_chat.status_code >= 400:
        raise RuntimeError(f"Public chat test failed: {public_chat.status_code} {public_chat.text[:1000]}")
    print("Public chat:", public_chat.json().get("message", {}).get("content", "")[:120])

print("\nAIクラウドに貼るURL:", PUBLIC_URL)
print("形式: Ollama")
print("モデル:", MODEL)

In [ ]:
# 任意: Colab内で動作確認
test = requests.post(
    f"http://127.0.0.1:{PORT}/api/chat",
    json={"model": MODEL, "messages": [{"role": "user", "content": "日本語で短くOKと返して"}], "stream": False},
    timeout=300,
)
print(test.status_code)
print(test.json().get("message", {}).get("content", test.text))